# SMC/ICT Strategy Backtest Analysis

This notebook demonstrates how to run the SMC/ICT reversal strategy
using the custom backtest engine and generate visualization reports.

**Note:** The SMC strategy requires 5-minute or finer intraday data.

---

## Quick Configuration Guide

Modify the `CONFIG` dictionary in Section 1 to customize:
- Data source and date range
- SMC strategy parameters (session times, ATR settings)
- Risk management parameters
- Output settings

---

## 1. Configuration Section

**Modify parameters below to customize the SMC backtest.**

In [ ]:
# ============================================================
# CONFIGURATION - Modify these parameters to customize analysis
# ============================================================

CONFIG = {
    # ----------------------------------------------------------
    # Data Configuration
    # ----------------------------------------------------------
    'data': {
        'file': 'SPY_5min.csv',            # Primary: 5-minute data
        'fallback_file': 'SPY_daily.csv',   # Fallback: daily data
        'directory': 'data/raw',
        'start_date': None,
        'end_date': None,
        'columns': ['Open', 'High', 'Low', 'Close', 'Volume'],
    },
    
    # ----------------------------------------------------------
    # SMC Strategy Configuration
    # ----------------------------------------------------------
    'smc': {
        # Session times (UTC)
        'session_start': '00:00',           # Asian session start
        'session_end': '08:00',             # Asian session end
        
        # ATR settings
        'atr_period': 14,
        'atr_buffer_mult': 0.5,
        'ifvg_atr_mult': 1.2,
        'ifvg_proximity_mult': 1.5,
        
        # Risk management
        'risk_per_trade': 0.01,            # 1% risk per trade
        'slippage_buffer': 0.1,
        
        # Targets
        'target_1r': 1.0,                    # First target at 1R (breakeven)
        'target_2r': 2.0,                   # Second target at 2R
        'target_final': 2.5,                # Final target at 2.5R
        
        # Daily limits
        'daily_loss_limit': 0.03,           # 3% daily loss limit
        'max_trades_per_day': 3,
        
        # Confirmations
        'require_volume_confirmation': True,
        'require_mss_confirmation': True,
    },
    
    # ----------------------------------------------------------
    # Backtest Configuration
    # ----------------------------------------------------------
    'backtest': {
        'initial_equity': 100000,
        'commission_pct': 0.001,            # 0.1% commission
        'slippage_pct': 0.0005,            # 0.05% slippage
        'risk_per_trade': 0.01,
        'max_open_positions': 1,           # SMC typically trades one position
        'min_confidence': 0.5,
    },
    
    # ----------------------------------------------------------
    # Output Configuration
    # ----------------------------------------------------------
    'output': {
        'directory': 'reports',
        'save_plots': True,
        'show_plots': True,
        'dpi': 150,
    },
}

---

## 2. Setup and Imports

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

# Import notebook helpers
from src.utils.notebook_helpers import (
    setup_project_root,
    load_price_data,
    print_data_summary,
    get_output_path,
)

# Setup project root
project_root = setup_project_root()

# Standard imports
import pandas as pd
import numpy as np
from datetime import datetime, time

# Import custom engine and SMC strategy
from src.backtest import BacktestEngine, BacktestConfig
from src.strategies import SMCReversalStrategy, SMCConfig
from src.indicators.asian_range import detect_asian_range
from src.indicators.ifvg import detect_ifvg
from src.indicators.mss import detect_mss

# Import visualization
from src.visualization import ReportGenerator, ChartGenerator

print("✅ Imports successful!")
print(f"Project root: {project_root}")

---

## 3. Load and Prepare Data

The SMC strategy requires intraday data (5-minute bars recommended).

In [ ]:
from pathlib import Path

# Try to load 5-minute data, fallback to daily
data_config = CONFIG['data']
data_path = project_root / data_config['directory'] / data_config['file']

if data_path.exists():
    df = load_price_data(CONFIG, project_root)
    data_freq = '5-minute'
    print(f"✅ Loaded 5-minute data")
else:
    print("⚠️ 5-minute data not found. Using daily data for demonstration.")
    print("For proper SMC backtesting, please provide 5-minute OHLCV data.")
    
    # Load daily data as fallback
    fallback_config = CONFIG.copy()
    fallback_config['data']['file'] = data_config['fallback_file']
    df = load_price_data(fallback_config, project_root)
    data_freq = 'daily'

# Display data summary
print_data_summary(df, title=f"SMC Backtest Data ({data_freq})")

# Check data frequency
median_diff = df.index.to_series().diff().median()
print(f"\nData frequency: {median_diff}")

---

## 4. SMC Strategy Components

Let's examine the SMC indicators individually.

In [ ]:
# Detect Asian Range (for 5-minute data)
# The Asian session is typically 00:00-08:00 UTC

if data_freq == '5-minute':
    print("Detecting Asian Range...")
    asian_range = detect_asian_range(df)
    if asian_range:
        print(f"Asian Range High: {asian_range.high}")
        print(f"Asian Range Low: {asian_range.low}")
        print(f"Range Size: {asian_range.range_size}")
        print(f"Is Low Volatility: {asian_range.is_low_vol}")
else:
    print("Daily data detected - Asian Range detection requires intraday data")

In [ ]:
# Detect IFVG (Inverse Fair Value Gaps)
print("Detecting IFVGs...")
ifvg_list = detect_ifvg(df)

print(f"Found {len(ifvg_list)} IFVGs")
if ifvg_list:
    print("\nRecent IFVGs:")
    for ifvg in ifvg_list[:5]:
        print(f"  Direction: {ifvg.direction}, Range: [{ifvg.low:.2f}, {ifvg.high:.2f}]")
        print(f"    Filled: {ifvg.filled}, Gap Size: {ifvg.gap_size:.2f}")

In [ ]:
# Detect MSS (Market Structure Shift)
print("Detecting Market Structure Shifts...")
mss_list = []

for i in range(50, len(df)):
    mss = detect_mss(df, i)
    if mss.detected and mss.is_valid:
        mss_list.append({
            'index': i,
            'timestamp': df.index[i],
            'direction': mss.direction,
            'break_price': mss.break_price
        })

print(f"Found {len(mss_list)} valid MSS signals")
if mss_list:
    print("\nRecent MSS signals:")
    for mss in mss_list[-5:]:
        print(f"  {mss['timestamp']}: {mss['direction']} at {mss['break_price']:.2f}")

---

## 5. Configure SMC Strategy

In [ ]:
# Create SMC configuration from CONFIG dict
smc_params = CONFIG['smc']

smc_config = SMCConfig(
    session_start=smc_params['session_start'],
    session_end=smc_params['session_end'],
    atr_period=smc_params['atr_period'],
    atr_buffer_mult=smc_params['atr_buffer_mult'],
    ifvg_atr_mult=smc_params['ifvg_atr_mult'],
    ifvg_proximity_mult=smc_params['ifvg_proximity_mult'],
    risk_per_trade=smc_params['risk_per_trade'],
    slippage_buffer=smc_params['slippage_buffer'],
    target_1r=smc_params['target_1r'],
    target_2r=smc_params['target_2r'],
    target_final=smc_params['target_final'],
    daily_loss_limit=smc_params['daily_loss_limit'],
    max_trades_per_day=smc_params['max_trades_per_day'],
    require_volume_confirmation=smc_params['require_volume_confirmation'],
    require_mss_confirmation=smc_params['require_mss_confirmation'],
)

print("SMC Strategy Configuration:")
print(f"  Session: {smc_config.session_start} - {smc_config.session_end} UTC")
print(f"  Risk per trade: {smc_config.risk_per_trade * 100}%")
print(f"  Daily loss limit: {smc_config.daily_loss_limit * 100}%")
print(f"  Max trades per day: {smc_config.max_trades_per_day}")

---

## 6. Run Backtest with Custom Engine

In [ ]:
# Configure backtest
backtest_params = CONFIG['backtest']

backtest_config = BacktestConfig(
    initial_equity=backtest_params['initial_equity'],
    commission_pct=backtest_params['commission_pct'],
    slippage_pct=backtest_params['slippage_pct'],
    risk_per_trade=backtest_params['risk_per_trade'],
    max_open_positions=backtest_params['max_open_positions'],
    min_confidence=backtest_params['min_confidence'],
)

print("Backtest Configuration:")
print(f"  Initial equity: ${backtest_config.initial_equity:,.0f}")
print(f"  Commission: {backtest_config.commission_pct * 100:.2f}%")
print(f"  Slippage: {backtest_config.slippage_pct * 100:.3f}%")

In [ ]:
# Initialize strategy
smc_strategy = SMCReversalStrategy(config=smc_config)

# Initialize backtest engine
engine = BacktestEngine(
    patterns=[smc_strategy],
    config=backtest_config
)

print("✅ Strategy and engine initialized")

In [ ]:
# Run backtest
print("Running SMC backtest...")

# Note: SMC strategy works best on 5-minute data
# Results on daily data will be limited

result = engine.run(
    df=df,
    start_date=None,  # Use all data
    end_date=None
)

print(f"\nBacktest complete!")
print(f"Total trades: {len(result.trades)}")
print(f"Total signals: {len(result.signals)}")

---

## 7. Analyze Results

In [ ]:
# Display metrics
if result.metrics:
    print("="*60)
    print("SMC STRATEGY PERFORMANCE")
    print("="*60)
    
    for key, value in result.metrics.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")
    print("="*60)

In [ ]:
# Display trades
if result.trades:
    print("\n📊 Recent Trades:")
    for trade in result.trades[:10]:
        print(f"  {trade}")
else:
    print("\n⚠️ No trades generated. This is expected with daily data.")
    print("   SMC strategy requires 5-minute or finer intraday data.")

In [ ]:
# Display signals
if result.signals:
    print(f"\n📊 Total signals generated: {len(result.signals)}")
    
    # Count by direction
    long_signals = sum(1 for s in result.signals if s.direction == 'long')
    short_signals = sum(1 for s in result.signals if s.direction == 'short')
    
    print(f"  Long signals: {long_signals}")
    print(f"  Short signals: {short_signals}")

---

## 8. Summary

In [ ]:
print("\n" + "="*60)
print("📊 SMC BACKTEST SUMMARY")
print("="*60)
print(f"\nData frequency: {data_freq}")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Total bars: {len(df):,}")

if data_freq == 'daily':
    print("\n⚠️ NOTE: SMC strategy is designed for intraday data.")
    print("   For meaningful results, please provide 5-minute OHLCV data.")
    print("   The strategy looks for:")
    print("   - Asian Range liquidity sweeps")
    print("   - Inverse Fair Value Gaps (IFVG)")
    print("   - Market Structure Shifts (MSS)")
else:
    print(f"\nTotal trades: {len(result.trades)}")
    print(f"Total signals: {len(result.signals)}")
    
    if result.metrics:
        print(f"\nKey Metrics:")
        print(f"  Total Return: {result.metrics.get('total_return', 'N/A')}")
        print(f"  Win Rate: {result.metrics.get('win_rate', 'N/A')}")
        print(f"  Sharpe Ratio: {result.metrics.get('sharpe_ratio', 'N/A')}")

print("\n" + "="*60)